# MLE deconvolver

The aim of this notebook is to implement and test different flavours of the MLE (maximul likelihood estimation) deconvolver

## Imports

In [1]:
from pathlib import Path
import warnings
from contextlib import contextmanager

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from tqdm.notebook import tqdm
from scipy.optimize import minimize

from edautils import plot_deconvolution_results
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics

In [2]:
@contextmanager
def SilenceValueOOBWarning():
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Values in x were outside bounds during a minimize step",
            category=RuntimeWarning,
        )
        yield

In [3]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
    ctype_names: list[str] = None,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label, alpha=0.7)

    # plot the cell types names on the x axis, rotated by 90 degrees
    if ctype_names is not None:
        plt.xticks(x_center, ctype_names, rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

## Data loading and preprocessing

Here we load the predictions for every read and we create 1000 pseudobulks in the same manner as the production pipeline.

In [ ]:
with open("../App/labels_dict.json", "r") as f:
    labels_dict = json.load(f)

### Data for soft labels

In [ ]:
# Load the test data for soft labels
test_reads_predictions_soft = pickle.load(open("../Data/training_data/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_j akkard_no_data_leak_d041/test_predicted.pkl", "rb"))

### Data for hard labels with focal loss

In [4]:
# load the predicted reads for the focal loss model
all_splits_predicted_reads_focal_loss = pickle.load(open("../Data/training_data/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_no_data_leak_d041_postfiltered_min_length_50_hard_labels_focal_loss_for_plotting/predicted_reads.pkl", "rb"))

### Data for hard labels with minibatch balancing

In [4]:
# load the predicted reads for the hard labels model (w/ background and minibatch balancing)
DATA_PATH = Path("../Data/training_data/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_minibatch_balanced")
train_reads_predictions = pickle.load(open(DATA_PATH / "train_predicted.pkl", "rb"))
test_reads_predictions = pickle.load(open(DATA_PATH / "test_predicted.pkl", "rb"))

In [ ]:
# we compute the average probit for on-target reads for each cell type
avg_on_target_probits = np.zeros(39)
for c in range(39):
    mask = (test_reads_predictions["original_label"] == c) & (test_reads_predictions["original_label"]==c)
    pred_col = "prediction_" + str(c)
    avg_on_target_probits[c] = test_reads_predictions.loc[mask, pred_col].mean()

Computing the train prior for hard labels with minibatch balancing:

Let :

- $\pi_c^{(S)}$ the proportions of signal (on-target) reads that are of ctype $c$
- $\pi_c^{(B)}$ the proportions of background (off-target) reads that are of ctype $c$
- $r=0.5$ the ratio of background reads in each minibatch
<!-- - $\alpha$ the fraction of signal read in the data
- $\pi_c = \alpha \pi_c^{(S)} + (1-\alpha) \pi_c^{(B)}$ the overall proportions of reads that are of ctype $c$ -->


Each batch sample $r$ reads from the background distribution and $1-r$ reads from the signal distribution.

Therefore the effective per class train proportion is :
$$
\hat{\pi}_c = r \pi_c^{(B)} + (1-r) \pi_c^{(S)}
$$

In [5]:
# P(c): prior from training data frequencies
train_class_counts = train_reads_predictions["original_label"].value_counts().sort_index()
train_signal_class_counts = train_reads_predictions[
    (train_reads_predictions["dmr_ctype_label"] == train_reads_predictions["original_label"])
]["original_label"].value_counts().sort_index()
train_background_class_counts = train_reads_predictions[
    (train_reads_predictions["dmr_ctype_label"] != train_reads_predictions["original_label"])
]["original_label"].value_counts().sort_index()
bg_ratio = 0.5
raw_train_freq_prior = (train_class_counts / train_class_counts.sum()).values
raw_train_signal_ratio = (train_signal_class_counts / train_signal_class_counts.sum()).values
raw_train_bg_ratio = (train_background_class_counts / train_background_class_counts.sum()).values
effective_train_prior = bg_ratio * raw_train_bg_ratio + (1 - bg_ratio) * raw_train_signal_ratio

## MLE (methylbert flavour)

In this cell, we describe mathematically the likelihood function that we want to maximize in MLE["methylbert flavour"].

Let $\{(r_i, c_i, g_i)\}_{i \in \{1, \dots, N\}}$ be the set of reads, where $r_i$ is the read, $c_i$ is the cell type, and $g_i$ is the DMR group.

Let $\{\mathbf{u}_i\}_{i \in \{1, \dots, n\}}$ be the set of predicted probabilities for every read and every cell type,
where $\mathbf{u}_i$ is a vector of length $C+1$ (number of cell types plus background class) and $u_{i,j}$ is the predicted probability that read $i$ belongs to class $j$.

We assume that the underlying true proportions of the cells are $\mathbf{\theta}= (\theta_1, \dots, \theta_C)$, where $\theta_c$ is the proportion of cell type $c$ in the sample.

Let $R_c = \{ i \mid g_i = c_i \}$ be the set of reads that fall in DMR group $c$.

Then the likelihood function that we want to maximize is given by:

\begin{align}
L(\mathbf{\theta}) &= \prod_{c=1}^C \prod_{i \in R_c} P(r_i \mid \theta) \\
&= \prod_{c=1}^C \prod_{i \in R_c} \left[\theta_c P(r_i \mid c) + (1-\theta_c) P(r_i \mid \text{not } c) \right] \\
&= \prod_{c=1}^C \prod_{i \in R_c} \left[\theta_c \frac{P(c \mid r_i)P(r_i)}{P(c)} + (1-\theta_c) \frac{P(\text{not } c \mid r_i)P(r_i)}{P(\text{not } c)} \right] \\
&= \prod_{c=1}^C \prod_{i \in R_c} P(r_i) \left[\theta_c \frac{P(c \mid r_i)}{P(c)} + (1-\theta_c) \frac{1 - P(c \mid r_i)}{1 - P(c)} \right] \\
\end{align}

where $P(c)$ is the prior probability of cell type $c$ in the sample, which we estimate as the proportions of the reads that belong to cell type $c$ in the training data (as in the paper).

In particular, this is NOT the equation 19 which is indicated in the paper, which is highly misleading:
\begin{align}
\tag{methylbert paper, equation 19}
L(\mathbf{\theta}) &= \prod_{c=1}^C \prod_{i \in R_c} \left( \theta_c P(r_i \mid c) + (1 - \theta_c) (1-P(r_i \mid c)) \right) \\
\end{align}



So the likelihood function can be rewritten as:

\begin{align}
L(\mathbf{\theta}) &= \prod_{c=1}^C \prod_{i \in R_c} P(r_i) \left[\theta_c \frac{u_{i,c}}{P(c)} + (1-\theta_c) \frac{1 - u_{i,c}}{1 - P(c)} \right] \\ \\
&= \prod_{c=1}^C \prod_{i \in R_c} P(r_i) \left[\theta_c ( \frac{u_{i,c}}{P(c)} - \frac{1 - u_{i,c}}{1 - P(c)}) + \frac{1 - u_{i,c}}{1 - P(c)} \right] \\ \\
&= \prod_{c=1}^C \prod_{i \in R_c} P(r_i) \left[\theta_c \frac{u_{i,c} - P(c)}{P(c)(1-P(c))} + \frac{1 - u_{i,c}}{1 - P(c)} \right] \\
\end{align}

and

$$
\log L(\mathbf{\theta}) = \sum_{c=1}^C \sum_{i \in R_c} \log \left( \theta_c \frac{u_{i,c} - P(c)}{P(c)(1-P(c))} + \frac{1 - u_{i,c}}{1 - P(c)} \right) +  \sum_{c=1}^C \sum_{i \in R_c} \log P(r_i)
$$


and

$$
\theta^* = \arg\max_{\mathbf{\theta}} \log L(\mathbf{\theta})
$$

In [5]:
from scipy.optimize import minimize

def _neg_log_likelihood(theta, coeffs_per_class, constants_per_class):
    """Negative log-likelihood for the methylbert MLE deconvolver."""
    nll = 0.0
    for c, (coeffs, constants) in enumerate(zip(coeffs_per_class, constants_per_class)):
        if coeffs is None:
            continue
        nll -= np.sum(np.log(theta[c] * coeffs + constants))
    return nll

def mle_deconvolver_methylbert_scipy(df: pd.DataFrame, P_c: np.ndarray, n_ctypes: int):
    """MLE deconvolver (methylbert flavour) using scipy.optimize.minimize.

    Maximizes the log-likelihood:
    subject to theta >= 0, sum(theta) = 1."""

    pred_cols = ["prediction_" + str(c) for c in range(n_ctypes)]

    # Precompute coefficients and constants per class
    coeffs_per_class = []
    constants_per_class = []
    for c in range(n_ctypes):
        mask = df["dmr_ctype_label"] == c
        if mask.sum() == 0:
            coeffs_per_class.append(None)
            constants_per_class.append(None)
            continue
        u_ic = df.loc[mask, pred_cols[c]].values
        # coeffs_per_class.append(2 * u_ic * P_c[c]**-1 - 1)
        # constants_per_class.append(1 - u_ic * P_c[c]**-1)
        coeffs_per_class.append((u_ic-P_c[c]) / (P_c[c]*(1-P_c[c])))
        constants_per_class.append((1 - u_ic) / (1-P_c[c]))

    result = minimize(
        _neg_log_likelihood,
        x0=np.ones(n_ctypes) / n_ctypes,
        args=(coeffs_per_class, constants_per_class),
        method="SLSQP",
        bounds=[(1e-10, 1 - 1e-10)] * n_ctypes,
        constraints={"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    return result.x

In [ ]:
# we want to see how much probability mass the model assigns outside of background or target cell type
n_ctypes = 39

proba_mass_outside_dmrtarget_or_background = pd.DataFrame({
    c: test_reads_predictions.loc[
        test_reads_predictions["dmr_ctype_label"] == c,
        [f"prediction_{k}" for k in range(n_ctypes) if k != c]
    ].sum(axis=1).describe()
    for c in range(n_ctypes)
})

proba_mass_outside_dmrtarget_or_background.columns = [f"dmr_ctype_{c}_{labels_dict[str(c)]}" for c in range(n_ctypes)]
proba_mass_outside_dmrtarget_or_background.T.sort_values("max", ascending=False)

### Experiments: MLE[methylbert flavour] and UXM on pure mixtures, on Methylbert outputs with focal loss

In [6]:
from methyldl.deconvolution.uxm import load_atlas, uxm_deconvolution, rearange_uxm_deconvolution_results

with open("../App/labels_dict.json", "r") as f:
    labels_dict = json.load(f)
labels_dict_reversed = {v: int(k) for k, v in labels_dict.items()}

# UXM atlas
uxm_atlas, ref_cells = load_atlas("../Data/UXM_atlas/Atlas.U25.l4.hg38.full.tsv")

# Effective train prior for the focal loss model
train_focal = all_splits_predicted_reads_focal_loss["train"]
test_focal = all_splits_predicted_reads_focal_loss["test"]
raw_train_freq_prior_fl = train_focal["original_label"].value_counts().sort_index() / len(train_focal)
# train_signal_counts_fl = train_focal[
#     train_focal["dmr_ctype_label"] == train_focal["original_label"]
# ]["original_label"].value_counts().sort_index()
# train_bg_counts_fl = train_focal[
#     train_focal["dmr_ctype_label"] != train_focal["original_label"]
# ]["original_label"].value_counts().sort_index()

# bg_ratio = 0.5
# effective_train_prior_fl = (
#     bg_ratio * (train_bg_counts_fl / train_bg_counts_fl.sum()).values
#     + (1 - bg_ratio) * (train_signal_counts_fl / train_signal_counts_fl.sum()).values
# )

# print(f"UXM atlas: {len(uxm_atlas)} regions, {len(ref_cells)} ref cell types")
# print(f"Effective train prior (focal loss): min={effective_train_prior_fl.min():.4f}, max={effective_train_prior_fl.max():.4f}")

In [7]:
n_ctypes = 39
target_proportions = np.eye(n_ctypes)
uniform_prior = np.ones(n_ctypes) / n_ctypes
all_predictions_fl = {}

# MLE methylbert with uniform prior
mle_uniform_preds = np.zeros((n_ctypes, n_ctypes))
for c in tqdm(range(n_ctypes), desc="MLE methylbert (uniform prior)"):
    pure_mixture = test_focal[test_focal["original_label"] == c]
    try:
        with SilenceValueOOBWarning():
            mle_uniform_preds[c] = mle_deconvolver_methylbert_scipy(pure_mixture, uniform_prior, n_ctypes)
    except Exception as e:
        print(f"Error for cell type {c}: {e}")
        mle_uniform_preds[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_fl["mle_methylbert_uniform"] = mle_uniform_preds

# MLE methylbert with raw train prior
mle_eff_preds = np.zeros((n_ctypes, n_ctypes))
for c in tqdm(range(n_ctypes), desc="MLE methylbert (raw train prior)"):
    pure_mixture = test_focal[test_focal["original_label"] == c]
    try:
        with SilenceValueOOBWarning():
            mle_eff_preds[c] = mle_deconvolver_methylbert_scipy(pure_mixture, raw_train_freq_prior_fl, n_ctypes)
    except Exception as e:
        print(f"Error for cell type {c}: {e}")
        mle_eff_preds[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_fl["mle_methylbert_raw_train"] = mle_eff_preds

# UXM
uxm_preds_fl = np.zeros((n_ctypes, n_ctypes))
sample_name = "pure_sample"
for c in tqdm(range(n_ctypes), desc="UXM deconvolution"):
    df_c = test_focal[test_focal["original_label"] == c].copy()
    if "direction" not in df_c.columns:
        df_c["direction"] = "U"
    agg = (
        df_c.groupby(["name", "direction"])
        .agg(record_M=("record_M", "sum"), record_U=("record_U", "sum"), record_X=("record_X", "sum"))
        .reset_index()
    )
    agg["count"] = agg["record_M"] + agg["record_U"] + agg["record_X"]
    agg = agg[agg["count"] > 0]
    sf = agg[["name", "direction"]].copy()
    sf[sample_name] = agg["record_U"] / agg["count"]
    counts = agg[["name", "direction"]].copy()
    counts[sample_name] = agg["count"]
    try:
        uxm_props = uxm_deconvolution(uxm_atlas, ref_cells, sf, counts, sample_names=[sample_name])[0]
        uxm_preds_fl[c] = rearange_uxm_deconvolution_results(labels_dict_reversed, uxm_props, ref_cells)
    except Exception as e:
        print(f"UXM error for cell type {c}: {e}")
        uxm_preds_fl[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_fl["uxm"] = uxm_preds_fl

MLE methylbert (uniform prior):   0%|          | 0/39 [00:00<?, ?it/s]

MLE methylbert (raw train prior):   0%|          | 0/39 [00:00<?, ?it/s]

UXM deconvolution:   0%|          | 0/39 [00:00<?, ?it/s]

In [8]:
# Compute and display metrics
class_names = [labels_dict[str(i)] for i in range(n_ctypes)]
deconv_display = {
    "mle_methylbert_uniform": "MLE[methylbert] (uniform prior)",
    "mle_methylbert_raw_train": "MLE[methylbert] (raw train prior)",
    "uxm": "UXM",
}

all_metrics_fl = {}
for name, pred in all_predictions_fl.items():
    m = compute_deconvolution_metrics(pred=pred, target=target_proportions, class_names=class_names)
    all_metrics_fl[name] = m
    print(f"{deconv_display[name]:45s}  R²={m['overall_r2']:.4f}  MAE={m['mae']:.6f}  MSE={m['mse']:.6f}  KL={m['kl']:.4f}  cosine_sim={m['cosine_sim']:.4f}")

MLE[methylbert] (uniform prior)                R²=-0.1869  MAE=0.049011  MSE=0.029654  KL=14.4671  cosine_sim=0.0855
MLE[methylbert] (raw train prior)              R²=-0.2092  MAE=0.049073  MSE=0.030209  KL=13.9151  cosine_sim=0.0788
UXM                                            R²=0.9817  MAE=0.004917  MSE=0.000457  KL=0.1081  cosine_sim=0.9980


In [9]:
# LaTeX table output
deconv_order_fl = ["mle_methylbert_uniform", "mle_methylbert_raw_train", "uxm"]
n_deconv = len(deconv_order_fl)

lines = []
for i, dec in enumerate(deconv_order_fl):
    m = all_metrics_fl[dec]
    r2 = f"{m['overall_r2'] * 100:.2f}"
    loa = f"[{m['loa_lower']*1e2:.2f}, {m['loa_upper']*1e2:.2f}]"
    loa_worst = f"[{m['worst_class_loa_lower']*1e2:.2f}, {m['worst_class_loa_upper']*1e2:.2f}]"
    mae = f"{m['mae']*1e3:.2f}"
    mse = f"{m['mse']*1e4:.2f}"
    kl = f"{m['kl']*1e2:.2f}"
    label = deconv_display[dec]

    if i == 0:
        lines.append(
            f"\\multirow{{{n_deconv}}}{{*}}{{Pure}}"
            f" & {label:<45s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )
    else:
        lines.append(
            f"{'':30s} & {label:<45s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )

    if i < n_deconv - 1:
        lines.append("\\cmidrule(l){2-8}")

print("\n".join(lines))

\multirow{3}{*}{Pure} & MLE[methylbert] (uniform prior)               & -18.69            & [-33.76, 33.76]     & [-4.50, 58.97]          & 49.01        & 296.54       & 1446.71 \\
\cmidrule(l){2-8}
                               & MLE[methylbert] (raw train prior)             & -20.92            & [-34.08, 34.08]     & [-14.25, 49.61]         & 49.07        & 302.09       & 1391.51 \\
\cmidrule(l){2-8}
                               & UXM                                           & 98.17             & [-4.21, 4.17]       & [-9.66, 9.00]           & 4.92         & 4.57         & 10.81 \\


In [10]:
for prior_type in ["uniform", "raw_train"]:    
    fl_avg_predictions_per_ctype = all_predictions_fl[f"mle_methylbert_{prior_type}"].mean(axis=0)
    majority_predicted_ctype = np.argmax(fl_avg_predictions_per_ctype)
    print(f"Majority predicted cell type across all pure samples ({prior_type} prior): {majority_predicted_ctype} ({labels_dict[str(majority_predicted_ctype)]}) with average predicted proportion {fl_avg_predictions_per_ctype[majority_predicted_ctype]:.4f}")

Majority predicted cell type across all pure samples (uniform prior): 6 (Blood-T) with average predicted proportion 0.2980
Majority predicted cell type across all pure samples (raw_train prior): 17 (Gallbladder) with average predicted proportion 0.4047


In [11]:
# bias cell type for UXM
uxm_avg_predictions_per_ctype = all_predictions_fl["uxm"].mean(axis=0)
uxm_majority_predicted_ctype = np.argmax(uxm_avg_predictions_per_ctype)
print(f"Majority predicted cell type across all pure samples (UXM): {uxm_majority_predicted_ctype} ({labels_dict[str(uxm_majority_predicted_ctype)]}) with average predicted proportion {uxm_avg_predictions_per_ctype[uxm_majority_predicted_ctype]:.4f}")

Majority predicted cell type across all pure samples (UXM): 6 (Blood-T) with average predicted proportion 0.0294


### Experiments: MLE[methylbert flavour] and UXM on pure mixtures, on Methylbert outputs with minibatch balancing

In [7]:
from methyldl.deconvolution.uxm import load_atlas, uxm_deconvolution, rearange_uxm_deconvolution_results

with open("../App/labels_dict.json", "r") as f:
    labels_dict = json.load(f)
labels_dict_reversed = {v: int(k) for k, v in labels_dict.items()}

# UXM atlas
uxm_atlas, ref_cells = load_atlas("../Data/UXM_atlas/Atlas.U25.l4.hg38.full.tsv")

n_ctypes = 39
target_proportions = np.eye(n_ctypes)
uniform_prior = np.ones(n_ctypes) / n_ctypes
all_predictions_mb = {}

# MLE methylbert with uniform prior
mle_uniform_preds = np.zeros((n_ctypes, n_ctypes))
for c in tqdm(range(n_ctypes), desc="MLE methylbert (uniform prior)"):
    pure_mixture = test_reads_predictions[test_reads_predictions["original_label"] == c]
    try:
        with SilenceValueOOBWarning():
            mle_uniform_preds[c] = mle_deconvolver_methylbert_scipy(pure_mixture, uniform_prior, n_ctypes)
    except Exception as e:
        print(f"Error for cell type {c}: {e}")
        mle_uniform_preds[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_mb["mle_methylbert_uniform"] = mle_uniform_preds

# MLE methylbert with effective train prior
mle_eff_preds = np.zeros((n_ctypes, n_ctypes))
for c in tqdm(range(n_ctypes), desc="MLE methylbert (effective train prior)"):
    pure_mixture = test_reads_predictions[test_reads_predictions["original_label"] == c]
    try:
        with SilenceValueOOBWarning():
            mle_eff_preds[c] = mle_deconvolver_methylbert_scipy(pure_mixture, effective_train_prior, n_ctypes)
    except Exception as e:
        print(f"Error for cell type {c}: {e}")
        mle_eff_preds[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_mb["mle_methylbert_effective_train"] = mle_eff_preds

# UXM
uxm_preds_mb = np.zeros((n_ctypes, n_ctypes))
sample_name = "pure_sample"
for c in tqdm(range(n_ctypes), desc="UXM deconvolution"):
    df_c = test_reads_predictions[test_reads_predictions["original_label"] == c].copy()
    if "direction" not in df_c.columns:
        df_c["direction"] = "U"
    agg = (
        df_c.groupby(["name", "direction"])
        .agg(record_M=("record_M", "sum"), record_U=("record_U", "sum"), record_X=("record_X", "sum"))
        .reset_index()
    )
    agg["count"] = agg["record_M"] + agg["record_U"] + agg["record_X"]
    agg = agg[agg["count"] > 0]
    sf = agg[["name", "direction"]].copy()
    sf[sample_name] = agg["record_U"] / agg["count"]
    counts = agg[["name", "direction"]].copy()
    counts[sample_name] = agg["count"]
    try:
        uxm_props = uxm_deconvolution(uxm_atlas, ref_cells, sf, counts, sample_names=[sample_name])[0]
        uxm_preds_mb[c] = rearange_uxm_deconvolution_results(labels_dict_reversed, uxm_props, ref_cells)
    except Exception as e:
        print(f"UXM error for cell type {c}: {e}")
        uxm_preds_mb[c] = np.ones(n_ctypes) / n_ctypes
all_predictions_mb["uxm"] = uxm_preds_mb

MLE methylbert (uniform prior):   0%|          | 0/39 [00:00<?, ?it/s]

MLE methylbert (effective train prior):   0%|          | 0/39 [00:00<?, ?it/s]

UXM deconvolution:   0%|          | 0/39 [00:00<?, ?it/s]

In [8]:
# Compute and display metrics
class_names = [labels_dict[str(i)] for i in range(n_ctypes)]
deconv_display_mb = {
    "mle_methylbert_uniform": "MLE[methylbert] (uniform prior)",
    "mle_methylbert_effective_train": "MLE[methylbert] (effective train prior)",
    "uxm": "UXM",
}

all_metrics_mb = {}
for name, pred in all_predictions_mb.items():
    m = compute_deconvolution_metrics(pred=pred, target=target_proportions, class_names=class_names)
    all_metrics_mb[name] = m
    print(f"{deconv_display_mb[name]:45s}  R²={m['overall_r2']:.4f}  MAE={m['mae']:.6f}  MSE={m['mse']:.6f}  KL={m['kl']:.4f}  cosine_sim={m['cosine_sim']:.4f}")

MLE[methylbert] (uniform prior)                R²=0.3430  MAE=0.036421  MSE=0.016414  KL=1.6417  cosine_sim=0.6078
MLE[methylbert] (effective train prior)        R²=0.5039  MAE=0.031310  MSE=0.012395  KL=1.4622  cosine_sim=0.7476
UXM                                            R²=0.9817  MAE=0.004917  MSE=0.000457  KL=0.1081  cosine_sim=0.9980


In [9]:
# LaTeX table output
deconv_order_mb = ["mle_methylbert_uniform", "mle_methylbert_effective_train", "uxm"]
n_deconv = len(deconv_order_mb)

lines = []
for i, dec in enumerate(deconv_order_mb):
    m = all_metrics_mb[dec]
    r2 = f"{m['overall_r2'] * 100:.2f}"
    loa = f"[{m['loa_lower']*1e2:.2f}, {m['loa_upper']*1e2:.2f}]"
    loa_worst = f"[{m['worst_class_loa_lower']*1e2:.2f}, {m['worst_class_loa_upper']*1e2:.2f}]"
    mae = f"{m['mae']*1e3:.2f}"
    mse = f"{m['mse']*1e4:.2f}"
    kl = f"{m['kl']*1e2:.2f}"
    label = deconv_display_mb[dec]

    if i == 0:
        lines.append(
            f"\\multirow{{{n_deconv}}}{{*}}{{Pure}}"
            f" & {label:<45s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )
    else:
        lines.append(
            f"{'':30s} & {label:<45s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )

    if i < n_deconv - 1:
        lines.append("\\cmidrule(l){2-8}")

print("\n".join(lines))

\multirow{3}{*}{Pure} & MLE[methylbert] (uniform prior)               & 34.30             & [-25.12, 25.12]     & [-33.95, 28.82]         & 36.42        & 164.14       & 164.17 \\
\cmidrule(l){2-8}
                               & MLE[methylbert] (effective train prior)       & 50.39             & [-21.83, 21.83]     & [-33.95, 28.82]         & 31.31        & 123.95       & 146.22 \\
\cmidrule(l){2-8}
                               & UXM                                           & 98.17             & [-4.21, 4.17]       & [-9.66, 9.00]           & 4.92         & 4.57         & 10.81 \\


In [15]:
for prior_type in ["uniform", "effective_train"]:    
    mb_avg_predictions_per_ctype = all_predictions_mb[f"mle_methylbert_{prior_type}"].mean(axis=0)
    majority_predicted_ctype = np.argmax(mb_avg_predictions_per_ctype)
    print(f"Majority predicted cell type across all pure samples ({prior_type} prior): {majority_predicted_ctype} ({labels_dict[str(majority_predicted_ctype)]}) with average predicted proportion {mb_avg_predictions_per_ctype[majority_predicted_ctype]:.4f}")

Majority predicted cell type across all pure samples (uniform prior): 26 (Neuron) with average predicted proportion 0.2859
Majority predicted cell type across all pure samples (effective_train prior): 26 (Neuron) with average predicted proportion 0.2555


## MLE (with soft labels)

With the same notations as above, we derive another formulation of the likelihood which 
is more general and takes into account the fact that the model outputs a probability
distribution over the cell types for each read, not just a single probability of the read belonging to the cell type of the DMR group it falls in.

\begin{align}
L'(\mathbf{\theta}) &= \prod_{i=1}^N P(r_i \mid \mathbf{\theta}) \\
&= \prod_{i=1}^N \sum_{c=1}^C \theta_c P(r_i \mid c) \\
&= \prod_{i=1}^N \sum_{c=1}^C \theta_c \frac{P(c \mid r_i)P(r_i)}{P(c)} \\
&= \prod_{i=1}^N P(r_i) \sum_{c=1}^C \theta_c \frac{u_{i,c}}{P(c)} \\
\end{align}

hence

\begin{align}
\log L'(\mathbf{\theta}) &= \sum_{i=1}^Nlog P(r_i) + \sum_{i=1}^N log \left( \sum_{c=1}^C \theta_c \frac{u_{i,c}}{P(c)} \right) \\
\theta^* &= \arg\max_{\mathbf{\theta}} \log L'(\mathbf{\theta})
\end{align}

In [ ]:
def _nllh_our(theta, coeffs):
    """Negative log-likelihood for our version of the MLE deconvolver."""
    return - np.sum(np.log(np.sum(coeffs * theta, axis=1)))
    

def mle_deconvolver_our_scipy(df: pd.DataFrame, P_c: np.ndarray, n_ctypes: int):
    """MLE deconvolver (our flavour) using scipy.optimize.minimize.

    Maximizes the log-likelihood
    subject to theta >= 0, sum(theta) = 1."""

    pred_cols = ["prediction_" + str(c) for c in range(n_ctypes)]

    # Precompute coefficients and constants per class per read
    u_ic_matrix = df[pred_cols].values  # shape (n_reads, n_ctypes)
    coeffs = u_ic_matrix / P_c  # shape (n_reads, n_ctypes)

    result = minimize(
        _nllh_our,
        x0=np.ones(n_ctypes) / n_ctypes,
        args=(coeffs, ),
        method="SLSQP",
        bounds=[(1e-10, 1 - 1e-10)] * n_ctypes,
        constraints={"type": "eq", "fun": lambda x: np.sum(x) - 1},
    )

    return result.x

In [ ]:
our_estimated_proportions = np.zeros((39, 39))
our_target_proportions = np.eye(39)
uniform_prior = np.ones(39) / 39
for c in tqdm(range(39)):
    # Create pure mixture of cell type c
    pure_mixture = test_reads_predictions_soft[test_reads_predictions_soft["original_label"] == c]
    try:
        with SilenceValueOOBWarning():
            est_props = mle_deconvolver_our_scipy(pure_mixture, effective_train_prior, 39)
    except Exception as e:
        print(f"Error for cell type {c}: {e}")
        est_props =  42 * np.ones(39)
    our_estimated_proportions[c] = est_props

In [ ]:
compute_deconvolution_metrics(
    pred=our_estimated_proportions,
    target=our_target_proportions,
    class_names = [labels_dict[str(i)] for i in range(39)],
)

Results on pure mixtures:

With train freq prior:
'mae': 0.0479658722616682,
'mse': 0.043427129134146265,
'kl': 4.881267106036938,
'max_error': 0.9999998628593519,
'cosine_sim': 0.08072092990577093,
'overall_r2': -0.738227984553592,
'loa_lower': -0.4085190967312068,
'loa_upper': 0.40864526944570534,
'loa_width': 0.8171643661769121,
'worst_class_idx': 17,
'worst_class_name': 'Gallbladder',
'worst_class_loa_lower': 0.28080356133566065,
'worst_class_loa_upper': 1.4068446566552422,
'worst_class_loa_width': 1.1260410953195814,


With uniform prior:
'mae': 0.030852812432050077,
'mse': 0.018162622913999112,
'kl': 2.1731952038654327,
'max_error': 0.9999999997821539,
'cosine_sim': 0.5604835003564489,
'overall_r2': 0.27301711967914055,
'loa_lower': -0.26423254198131746,
'loa_upper': 0.26423481111087416,
'loa_width': 0.5284673530921916,
'worst_class_idx': 13,
'worst_class_name': 'Endothel',
'worst_class_loa_lower': 0.13192625407729736,
'worst_class_loa_upper': 0.8792231611067727,
'worst_class_loa_width': 0.7472969070294754,

With effective train freq prior:
'mae': 0.047928767481941165,
 'mse': 0.04280314711713293,
 'kl': 5.614476815095233,
 'max_error': 0.9999999998963527,
 'cosine_sim': 0.08181892223889449,
 'overall_r2': -0.7132522832936632,
 'loa_lower': -0.4055926783006242,
 'loa_upper': 0.40567975929336453,
 'loa_width': 0.8112724375939887,
 'worst_class_idx': 12,
 'worst_class_name': 'Dermal-Fibro',
 'worst_class_loa_lower': 0.21799052876841374,
 'worst_class_loa_upper': 1.4056639514085458,
 'worst_class_loa_width': 1.1876734226401322

In [ ]:
focus_ctype = 14
plot_mixtures_pred_vs_true(
    ground_truth_mixture=our_target_proportions[focus_ctype],
    predicted_mixtures=[our_estimated_proportions[focus_ctype]],
    predicted_mixture_labels=["MLE Deconvolver (our style)"],
    title=f"Predicted Mixture vs Ground Truth for Cell Type {focus_ctype} ({labels_dict[str(focus_ctype)]})",
    ctype_names=[labels_dict[str(i)] for i in range(39)],
)

### All deconvolvers comparison on pure mixtures (soft labels)

In [ ]:
import torch
import torch.nn as nn
from methyldl.deconvolution.least_squares_deconvolvers import NNLSDeconvolver, PSLSDeconvolver
from methyldl.deconvolution.xgbdeconvolver import XGBoostDeconvolver
from methyldl.deconvolution.feature_selection import apply_feature_mask
from methyldl.deconvolution.uxm import load_atlas, uxm_deconvolution, rearange_uxm_deconvolution_results

SOFT_DATA_PATH = Path(
    "../Data/training_data/"
    "methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10"
    "_postfiltered_min_length_50_soft_labels_pooled_j akkard_no_data_leak_d041"
)
DECONV_PATH = SOFT_DATA_PATH / "fitted_deconvolvers"

# Feature mask
feature_mask = np.load(DECONV_PATH / "features_mask_1.094806660557914_cutoff.npz")["features_mask"]

# Load deconvolvers
xgb_model = XGBoostDeconvolver.load(str(DECONV_PATH / "xgb_deconvolver.joblib"))
nnls_model = NNLSDeconvolver.load(str(DECONV_PATH / "nnls_deconvolver.joblib"))
psls_model = PSLSDeconvolver.load(str(DECONV_PATH / "psls_deconvolver.joblib"))

# NN models
def _build_nn(name, meta):
    n_in, n_out = meta["n_input_features"], meta["n_cell_types"]
    p = meta["params"]
    if name == "swn":
        hd = p.get("hidden_dim", 1024)
        return nn.Sequential(
            nn.Linear(n_in, hd), nn.GELU(), nn.Dropout(p.get("dropout", 0.2)),
            nn.Linear(hd, n_out), nn.Softmax(dim=-1),
        )
    elif name == "mlp":
        layers = []
        dim = n_in
        for h in p.get("hidden_dims", [512, 256]):
            layers += [nn.Linear(dim, h), nn.GELU(), nn.Dropout(p.get("dropout", 0.2))]
            dim = h
        layers += [nn.Linear(dim, n_in), nn.GELU(), nn.Dropout(p.get("final_dropout", 0.1))]
        layers += [nn.Linear(n_in, n_out), nn.Softmax(dim=-1)]
        return nn.Sequential(*layers)

with open(DECONV_PATH / "mlp_architecture_meta.json") as f:
    mlp_meta = json.load(f)
with open(DECONV_PATH / "swn_architecture_meta.json") as f:
    swn_meta = json.load(f)

mlp_model = _build_nn("mlp", mlp_meta)
mlp_model.load_state_dict(torch.load(DECONV_PATH / "mlp_best_deconvolver.pt", map_location="cpu", weights_only=True))
mlp_model.eval()

swn_model = _build_nn("swn", swn_meta)
swn_model.load_state_dict(torch.load(DECONV_PATH / "swn_best_deconvolver.pt", map_location="cpu", weights_only=True))
swn_model.eval()

# UXM atlas
uxm_atlas, ref_cells = load_atlas("../Data/UXM_atlas/Atlas.U25.l4.hg38.full.tsv")
labels_dict_reversed = {v: int(k) for k, v in labels_dict.items()}

print(f"Feature mask: {feature_mask.shape}, {int(feature_mask.sum())} features selected")
print(f"UXM atlas: {len(uxm_atlas)} regions, {len(ref_cells)} ref cell types")
print("Loaded: XGB, NNLS, PSLS, MLP, SWN, UXM atlas")

In [ ]:
# Compute feature matrices for pure mixtures (CpG-weighted avg per DMR group)
n_ctypes = 39
pred_cols = [f"prediction_{k}" for k in range(n_ctypes)]
weight_col = "total_marked_cpgs" if "total_marked_cpgs" in test_reads_predictions_soft.columns else "NCPGS"

pure_feature_matrix = np.zeros((n_ctypes, n_ctypes, n_ctypes))
for c in tqdm(range(n_ctypes), desc="Computing feature matrices"):
    df_c = test_reads_predictions_soft[test_reads_predictions_soft["original_label"] == c]
    for dmr_g, group in df_c.groupby("dmr_ctype_label"):
        dmr_g = int(dmr_g)
        if dmr_g >= n_ctypes:
            continue
        w = group[weight_col].values.astype(float).clip(min=1e-10)
        pure_feature_matrix[c, dmr_g] = np.average(group[pred_cols].values, weights=w, axis=0)

# Apply feature mask
pure_features_masked = apply_feature_mask(pure_feature_matrix, feature_mask)
print(f"Feature matrix shape: {pure_feature_matrix.shape}")
print(f"Masked features shape: {pure_features_masked.shape}")

In [ ]:
# Apply all deconvolvers
target_proportions = np.eye(n_ctypes)
all_predictions = {}

# XGB
xgb_raw = xgb_model._predict_raw(pure_features_masked)
all_predictions["xgb"] = xgb_model._transform_output(xgb_raw)

# MLP
with torch.no_grad():
    all_predictions["mlp"] = mlp_model(torch.FloatTensor(pure_features_masked)).numpy()

# SWN
with torch.no_grad():
    all_predictions["swn"] = swn_model(torch.FloatTensor(pure_features_masked)).numpy()

# NNLS
nnls_pred, _, _ = nnls_model.predict(pure_features_masked, n_workers=1)
all_predictions["nnls"] = nnls_pred

# PSLS
all_predictions["psls"] = psls_model.predict(pure_features_masked, n_workers=2)

# # MLE train prior
# mle_train_prior_preds = np.zeros((n_ctypes, n_ctypes))
# for c in tqdm(range(n_ctypes), desc="MLE deconvolution (train freq prior)"):
#     pure_mixture = test_reads_predictions_soft[test_reads_predictions_soft["original_label"] == c]
#     try:
#         with SilenceValueOOBWarning():
#             mle_train_prior_preds[c] = mle_deconvolver_our_scipy(pure_mixture, raw_train_freq_prior, n_ctypes)
#     except Exception as e:
#         print(f"Error for cell type {c}: {e}")
#         mle_train_prior_preds[c] = np.ones(n_ctypes) / n_ctypes
# all_predictions["mle_train_prior"] = mle_train_prior_preds

# # MLE uniform
# mle_uniform_prior_preds = np.zeros((n_ctypes, n_ctypes))
# for c in tqdm(range(n_ctypes), desc="MLE deconvolution (uniform prior)"):
#     pure_mixture = test_reads_predictions_soft[test_reads_predictions_soft["original_label"] == c]
#     try:
#         with SilenceValueOOBWarning():
#             mle_uniform_prior_preds[c] = mle_deconvolver_our_scipy(pure_mixture, np.ones(n_ctypes) / n_ctypes, n_ctypes)
#     except Exception as e:
#         print(f"Error for cell type {c}: {e}")
#         mle_uniform_prior_preds[c] = np.ones(n_ctypes) / n_ctypes
# all_predictions["mle_uniform_prior"] = mle_uniform_prior_preds

# UXM
uxm_preds = np.zeros((n_ctypes, n_ctypes))
sample_name = "pure_sample"
for c in tqdm(range(n_ctypes), desc="UXM deconvolution"):
    df_c = test_reads_predictions_soft[test_reads_predictions_soft["original_label"] == c].copy()
    if "direction" not in df_c.columns:
        df_c["direction"] = "U"
    agg = (
        df_c.groupby(["name", "direction"])
        .agg(record_M=("record_M", "sum"), record_U=("record_U", "sum"), record_X=("record_X", "sum"))
        .reset_index()
    )
    agg["count"] = agg["record_M"] + agg["record_U"] + agg["record_X"]
    agg = agg[agg["count"] > 0]
    sf = agg[["name", "direction"]].copy()
    sf[sample_name] = agg["record_U"] / agg["count"]
    counts = agg[["name", "direction"]].copy()
    counts[sample_name] = agg["count"]
    try:
        uxm_props = uxm_deconvolution(uxm_atlas, ref_cells, sf, counts, sample_names=[sample_name])[0]
        uxm_preds[c] = rearange_uxm_deconvolution_results(labels_dict_reversed, uxm_props, ref_cells)
    except Exception as e:
        print(f"UXM error for cell type {c}: {e}")
        uxm_preds[c] = np.ones(n_ctypes) / n_ctypes
all_predictions["uxm"] = uxm_preds

# Compute metrics
class_names = [labels_dict[str(i)] for i in range(n_ctypes)]
all_metrics = {}
for name, pred in all_predictions.items():
    m = compute_deconvolution_metrics(pred=pred, target=target_proportions, class_names=class_names)
    all_metrics[name] = m
    print(f"{name:20s}  R²={m['overall_r2']:.4f}  MAE={m['mae']:.6f}  MSE={m['mse']:.6f}  KL={m['kl']:.4f}")

In [ ]:
test_reads_predictions_soft.columns

In [ ]:
# LaTeX table output
deconv_map = {"xgb": "XGB", "mlp": "MLP", "swn": "SWN", "nnls": "NNLS", "psls": "PSLS", "mle_train_prior": "MLE (train prior)", "mle_uniform_prior": "MLE (uniform prior)", "uxm": "UXM"}
deconv_order = ["xgb", "mlp", "swn", "nnls", "psls", "uxm"]#"mle_train_prior", "mle_uniform_prior", 
n_deconv = len(deconv_order)

lines = []
for i, dec in enumerate(deconv_order):
    m = all_metrics[dec]
    r2 = f"{m['overall_r2'] * 100:.2f}"
    loa = f"[{m['loa_lower']*1e2:.2f}, {m['loa_upper']*1e2:.2f}]"
    loa_worst = f"[{m['worst_class_loa_lower']*1e2:.2f}, {m['worst_class_loa_upper']*1e2:.2f}]"
    mae = f"{m['mae']*1e3:.2f}"
    mse = f"{m['mse']*1e4:.2f}"
    kl = f"{m['kl']*1e2:.2f}"
    label = deconv_map[dec]

    if i == 0:
        lines.append(
            f"\\multirow{{{n_deconv}}}{{*}}{{Pure}}"
            f" & {label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )
    else:
        lines.append(
            f"{'':30s} & {label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
        )

    if i < n_deconv - 1:
        lines.append("\\cmidrule(l){2-8}")

print("\n".join(lines))